# Fluorescence Lifetime Decay Analysis - Calibration Data Processing

This notebook processes Time-Tagged Time-Resolved (TTTR) data files (.ptu format) to extract fluorescence decay curves and analyze photon statistics. The workflow is designed for PIE (Pulsed Interleaved Excitation) FLIM measurements with dual-color detection.

**Author:** Katherina Hemmen ~ Core Unit Fluorescence Imaging ~ RVZ

**Contact:** katherina.hemmen@uni-wuerzburg.de

**Tested with:** tttrlib 0.25.0

## Workflow Overview:
1. **Export decay histograms** - Convert raw TTTR data to fluorescence decay curves
2. **Calculate photon statistics** - Extract background levels and photon counts

## Output Files:
- `.txt` files: Decay data compatible with ChiSurf & other analysis software, containing the stacked histograms [parallel - pependicular channel]
- `summary.txt` file: Summarizes the background levels, total photons counts and ROI size (in pixel)

In [1]:
# Import required libraries for TTTR data analysis and visualization
import numpy as np  # Numerical operations and array handling
import pylab as p   # Plotting and visualization
import tttrlib      # Time-tagged time-resolved data analysis library
import glob         # File path pattern matching
import os           # Operating system interface

## 1. Export the Decay Curves

This section processes TTTR files to create fluorescence decay histograms. The data uses PIE (Pulsed Interleaved Excitation) with alternating green and red excitation pulses within each laser period.

In [2]:
# Define the path to calibration data files
# Uses glob pattern to process all .ptu files in the Calibration folder

path = 'C:/Users/Katherina Hemmen/Desktop/ExampleData/Raw_Files/Calibration/*.ptu'

### MEASUREMENT CONFIGURATION

In [3]:
# Channel assignments for PIE-FLIM setup
# Note: PIE (Pulsed Interleaved Excitation) alternates green and red excitation
green_ch_s = [0]  # Green perpendicular polarization channel
green_ch_p = [3]  # Green parallel polarization channel
red_ch_s = [1]    # Red perpendicular polarization channel
red_ch_p = [2]    # Red parallel polarization channel

# Time window definitions for PIE measurement
# Total time window = 50 ns, TCSPC bin size = 10 ps → 5000 bins total
# PIE splits this into two 25 ns windows (2500 bins each)
prompt_range = 0, 2500      # First half: green excitation + green/red prompt emission
delay_range = 2500, 5000    # Second half: red excitation + red delayed emission

# Histogram binning factor
# 1 = no binning (full resolution), >1 = reduce resolution by rebinning channels
binning = 2  # Reduces histogram resolution by factor of 2

### MAIN PROCESSING LOOP

Process each .ptu file in the specified directory

In [4]:
for file in glob.glob(path):
    # FILE LOADING AND HEADER EXTRACTION
    # Extract filename without extension for output file naming
    filename = os.path.abspath(file).split(".ptu")[0]
    
    # Load TTTR data file
    data = tttrlib.TTTR(file)
    
    # Extract timing calibration parameters from file header
    header = data.header
    macro_time_calibration = data.header.macro_time_resolution  # Laser period in nanoseconds
    micro_times = data.micro_times                              # Arrival times within laser period
    micro_time_resolution = data.header.micro_time_resolution   # Time resolution in seconds
    
    # Calculate histogram parameters
    expected_nr_of_bins = int(macro_time_calibration / micro_time_resolution) # Total bins in full time window
    binned_nr_of_bins = int(expected_nr_of_bins / binning)   # Bins after rebinning
    
    # CHANNEL SELECTION
    # Extract photon indices for each detection channel
    green_s_indices = data.get_selection_by_channel(green_ch_s)  # Green perpendicular
    green_p_indices = data.get_selection_by_channel(green_ch_p)  # Green parallel
    red_s_indices = data.get_selection_by_channel(red_ch_s)      # Red perpendicular
    red_p_indices = data.get_selection_by_channel(red_ch_p)      # Red parallel
    # Extract micro-times (arrival times) for selected photons
    green_s = micro_times[green_s_indices]
    green_p = micro_times[green_p_indices]
    red_s = micro_times[red_s_indices]
    red_p = micro_times[red_p_indices]
    
    # PIE WEIGHT GENERATION
    # Create binary weights to separate prompt and delay time windows
    # PIE splits each laser period in half: first half for green+red_prompt, second half for red_delay
    # Green channels: only count photons in first half (prompt window)
    w_prompt_green_s = np.ones_like(green_s, dtype=np.int64)
    w_prompt_green_s[np.where(green_s > expected_nr_of_bins // 2)] *= 0  # Zero out second half
    w_prompt_green_p = np.ones_like(green_p, dtype=np.int64)
    w_prompt_green_p[np.where(green_p > expected_nr_of_bins // 2)] *= 0
    
    # Red prompt channels: only count photons in first half
    w_prompt_red_s = np.ones_like(red_s, dtype=np.int64)
    w_prompt_red_s[np.where(red_s > expected_nr_of_bins // 2)] *= 0
    w_prompt_red_p = np.ones_like(red_p, dtype=np.int64)
    w_prompt_red_p[np.where(red_p > expected_nr_of_bins // 2)] *= 0
    
    # Red delay channels: only count photons in second half
    w_delay_red_s = np.ones_like(red_s, dtype=np.int64)
    w_delay_red_s[np.where(red_s < expected_nr_of_bins // 2)] *= 0  # Zero out first half
    w_delay_red_p = np.ones_like(red_p, dtype=np.int64)
    w_delay_red_p[np.where(red_p < expected_nr_of_bins // 2)] *= 0
    
    # HISTOGRAM CONSTRUCTION
    # Build weighted histograms for each channel and time window
    # np.bincount creates histograms with specified weights and minimum length
    green_s_counts = np.bincount(green_s // binning, weights=w_prompt_green_s, minlength=binned_nr_of_bins)
    green_p_counts = np.bincount(green_p // binning, weights=w_prompt_green_p, minlength=binned_nr_of_bins)
    red_s_counts_prompt = np.bincount(red_s // binning, weights=w_prompt_red_s, minlength=binned_nr_of_bins)
    red_p_counts_prompt = np.bincount(red_p // binning, weights=w_prompt_red_p, minlength=binned_nr_of_bins)
    red_s_counts_delay = np.bincount(red_s // binning, weights=w_delay_red_s, minlength=binned_nr_of_bins)
    red_p_counts_delay = np.bincount(red_p // binning, weights=w_delay_red_p, minlength=binned_nr_of_bins)
    
    # HISTOGRAM TRIMMING
    # Address potential rounding errors that create extra bins beyond expected length
    # Trim histograms to exact expected dimensions
    green_s_counts_cut = green_s_counts[0:binned_nr_of_bins :]
    green_p_counts_cut = green_p_counts[0:binned_nr_of_bins :]
    red_s_counts_cut_prompt = red_s_counts_prompt[0:binned_nr_of_bins // 2:]
    red_p_counts_cut_prompt = red_p_counts_prompt[0:binned_nr_of_bins // 2:]
    red_s_counts_cut_delay = red_s_counts_delay[binned_nr_of_bins // 2:binned_nr_of_bins:]
    red_p_counts_cut_delay = red_p_counts_delay[binned_nr_of_bins // 2:binned_nr_of_bins:]
    
    # FORMAT CONVERSION FOR EXTERNAL ANALYSIS SOFTWARE
    # Concatenate parallel and perpendicular channels for each color/time window
    # Format: [parallel_channel, perpendicular_channel] - compatible with e.g. with Seidel-Package, ChiSurf
    jordi_counts_green = np.concatenate([green_p_counts_cut[0:1250], green_s_counts_cut[0:1250]])
    jordi_counts_red_prompt = np.concatenate([red_p_counts_cut_prompt, red_s_counts_cut_prompt])
    jordi_counts_red_delay = np.concatenate([red_p_counts_cut_delay, red_s_counts_cut_delay])
    
    # SAVE DECAY HISTOGRAMS
    # Export processed decay curves as text files for further analysis
    # Save green decay curve
    jordi_format_green = filename + "_green_jordi.txt"
    np.savetxt(
        jordi_format_green,
         np.vstack([jordi_counts_green]).T  # Transpose to create column format
    )
    # Save red prompt decay curve
    jordi_format_red_prompt = filename + "_red_prompt_jordi.txt"
    np.savetxt(
        jordi_format_red_prompt,
        np.vstack([jordi_counts_red_prompt]).T
    )
    # Save red delay decay curve
    jordi_format_red_delay = filename + "_red_delay_jordi.txt"
    np.savetxt(
        jordi_format_red_delay,
        np.vstack([jordi_counts_red_delay]).T
    )    

## 2. Export Photon Counts and Background Analysis

This section analyzes the saved decay curves to extract:
- **Background levels**: Measured from early time bins (before fluorescence signal)
- **Total photon counts**: Sum of all photons in each decay curve
- **ROI size**: in pixel, for count rate calculation/area normalization

In [5]:
# Define input and output file paths

path = 'C:/Users/Katherina Hemmen/temp/ExampleData/Calibration/*.ptu' # Input: original TTTR files
save_file_as = './Summary.txt' # Output: analysis results summary

### INITIALIZE DATA STORAGE LISTS
Create empty lists to accumulate analysis results from all files

In [6]:
list_filenames = list()                # Store filename for each measurement

# Green channel analysis results
list_bg_s_green = list()               # Background level - perpendicular channel
list_bg_p_green = list()               # Background level - parallel channel
list_nr_photons_s_green = list()       # Total photon count - perpendicular channel
list_nr_photons_p_green = list()       # Total photon count - parallel channel

# Red prompt window analysis results
list_bg_s_red_prompt = list()          # Background level - perpendicular channel
list_bg_p_red_prompt = list()          # Background level - parallel channel
list_nr_photons_s_red_prompt = list()  # Total photon count - perpendicular channel
list_nr_photons_p_red_prompt = list()  # Total photon count - parallel channel

# Red delay window analysis results
list_bg_s_red_delay = list()           # Background level - perpendicular channel
list_bg_p_red_delay = list()           # Background level - parallel channel
list_nr_photons_s_red_delay = list()   # Total photon count - perpendicular channel
list_nr_photons_p_red_delay = list()   # Total photon count - parallel channel

list_nr_pixel_in_roi = list()          # Number of pixels in region of interest (if applicable)

### ANALYZE EACH DECAY CURVE FILE

In [7]:
for file in glob.glob(path):
    # Generate filenames for the decay curve files created in previous section
    filename = os.path.abspath(file).split(".")[0]        # Remove file extension
    data_green = filename  + '_green_jordi.txt'           # Green decay curve file
    data_red_delay = filename + '_red_delay_jordi.txt'    # Red delay decay curve file
    data_red_prompt = filename + '_red_prompt_jordi.txt'  # Red prompt decay curve file
    
    # LOAD DECAY CURVE DATA
    # Load the previously saved decay histograms
    decay_green = np.loadtxt(data_green, skiprows=0)
    decay_red_delay = np.loadtxt(data_red_delay, skiprows=0)
    decay_red_prompt = np.loadtxt(data_red_prompt, skiprows=0)
    
    # BACKGROUND CALCULATION
    # Background is measured from early time bins (before significant fluorescence signal)
    # Time bins 1-75 represent the baseline level before fluorescence starts to rise
    
    # Parallel channel backgrounds (first half of each decay array)
    bg_p_green = np.mean(decay_green[1:75])            # Green parallel background
    bg_p_red_delay = np.mean(decay_red_delay[1:75])    # Red delay parallel background
    bg_p_red_prompt = np.mean(decay_red_prompt[1:75])  # Red prompt parallel background
    
    # Perpendicular channel backgrounds (second half of each decay array)
    # Note: Different indexing reflects the concatenated structure [parallel, perpendicular]
    bg_s_green = np.mean(decay_green[1251:1325])            # Green perpendicular background  
    bg_s_red_delay = np.mean(decay_red_delay[1251:1325])    # Red delay perpendicular background
    bg_s_red_prompt = np.mean(decay_red_prompt[1251:1325])  # Red prompt perpendicular background
    
    # TOTAL PHOTON COUNT CALCULATION
    # Sum all photons in each decay curve for signal strength assessment
    
    # Green channel photon counts
    sum_p_green = np.sum(decay_green[0:1250])     # Parallel channel (first 1250 bins)
    sum_s_green = np.sum(decay_green[1250:2500])  # Perpendicular channel (last 1250 bins)
    # Red delay channel photon counts 
    sum_p_red_delay = np.sum(decay_red_delay[0:1250])     # Parallel channel (first 1250 bins)
    sum_s_red_delay = np.sum(decay_red_delay[1250:2500])  # Perpendicular channel (last 1250 bins)
    
    sum_p_red_prompt = np.sum(decay_red_prompt[0:1250])     # Parallel channel (first 1250 bins)
    sum_s_red_prompt = np.sum(decay_red_prompt[1250:2500])  # Perpendicular channel (last 1250 bins)

    # STORE RESULTS IN GROWING LISTS
    # Append calculated values to respective lists for final export
    list_filenames.append(str(file))
    
    # Green channel results
    list_bg_p_green.append(bg_p_green)
    list_nr_photons_p_green.append(sum_p_green)
    list_bg_s_green.append(bg_s_green)
    list_nr_photons_s_green.append(sum_s_green)
    
    # Red prompt results
    list_bg_s_red_prompt.append(bg_s_red_prompt)
    list_bg_p_red_prompt.append(bg_p_red_prompt)
    list_nr_photons_s_red_prompt.append(sum_s_red_prompt)
    list_nr_photons_p_red_prompt.append(sum_p_red_prompt)
    
    # Red delay results
    list_bg_s_red_delay.append(bg_s_red_delay)
    list_bg_p_red_delay.append(bg_p_red_delay)
    list_nr_photons_s_red_delay.append(sum_s_red_delay)
    list_nr_photons_p_red_delay.append(sum_p_red_delay)
    
    # EXPORT COMPREHENSIVE ANALYSIS RESULTS
    # Create descriptive column headers for the output file
    header = 'Filename\tBGgreen(p)\tNrPhotgreen(p)\tBGgreen(s)\tNrPhotgreen(s)\tBGrp(p)\tNrPhotrp(p)\tBGrp(s)\tNrPhotrp(s)\tBGrd(p)\tNrPhotrd(p)\tBGrd(s)\tNrPhotrd(s)'

    # Save all analysis results as tab-delimited text file
    # Format: rows = files, columns = analysis parameters
    np.savetxt(
        save_file_as,
        np.vstack(
            [
            list_filenames,                 # File identifiers
            list_bg_p_green,                # Green backgrounds & counts
            list_nr_photons_p_green,
            list_bg_s_green,
            list_nr_photons_s_green,
            list_bg_p_red_prompt,           # Red prompt backgrounds & counts
            list_nr_photons_p_red_prompt,
            list_bg_s_red_prompt,
            list_nr_photons_s_red_prompt,
            list_bg_p_red_delay,            # Red delay backgrounds & counts
            list_nr_photons_p_red_delay,
            list_bg_s_red_delay,
            list_nr_photons_s_red_delay
            ],
        ).T,               # Transpose for row-wise file organization
        delimiter='\t',    # Tab-separated format
        fmt="%s",          # String format for all columns
        header=header      # Include descriptive header
    )